In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

TRAIN_PATH = "/Users/nepulseneviratne/Downloads/dataset-2.csv"
TARGET_COLUMN = "price"
EVALUATION_SPLIT_PATH = "./evaluation_split_task2.csv"

## 1. Load the Data

In [17]:
df = pd.read_csv(TRAIN_PATH)
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,BMW,M235 i,2015,"81,000 mi.",Gasoline,320.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,6-Speed M/T,White,Red,None reported,Yes,"$27,890"
1,BMW,750 i,2017,"24,650 mi.",Gasoline,445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Black,None reported,Yes,"$50,000"
2,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"
3,INFINITI,QX80 Luxe,2022,"49,860 mi.",Gasoline,5.6L V8 32V GDI DOHC,Automatic,Black Obsidian,Graphite,At least 1 accident or damage reported,Yes,"$47,645"
4,Subaru,BRZ Premium,2023,"5,800 mi.",Gasoline,228.0HP 2.4L 4 Cylinder Engine Gasoline Fuel,6-Speed M/T,Black,Black,None reported,Yes,"$29,990"


## 2. Explore and Clean the Dataset


In [18]:
df.info()
print("Duplicates:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 3207 entries, 0 to 3206
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         3207 non-null   str  
 1   model         3207 non-null   str  
 2   model_year    3207 non-null   int64
 3   milage        3207 non-null   str  
 4   fuel_type     3081 non-null   str  
 5   engine        3207 non-null   str  
 6   transmission  3207 non-null   str  
 7   ext_col       3207 non-null   str  
 8   int_col       3207 non-null   str  
 9   accident      3115 non-null   str  
 10  clean_title   2728 non-null   str  
 11  price         3207 non-null   str  
dtypes: int64(1), str(11)
memory usage: 714.1 KB
Duplicates: 0


In [19]:
def to_number(series):
    return pd.to_numeric(series.astype(str).str.replace(r"[^\d.]", "", regex=True), errors="coerce")


def data_cleaning(df):
    df = df.copy()
    df["milage"] = to_number(df["milage"])
    df["fuel_type"] = df["fuel_type"].replace(["–", "not supported"], np.nan)
    electric = df["fuel_type"].isna() & df["engine"].astype(str).str.contains("Electric", case=False)
    df.loc[electric, "fuel_type"] = "Electric"
    df["fuel_type"] = df["fuel_type"].fillna("Unknown")
    if TARGET_COLUMN in df:
        df[TARGET_COLUMN] = to_number(df[TARGET_COLUMN])
        df = df.drop_duplicates().dropna(subset=[TARGET_COLUMN])
        low, high = df[TARGET_COLUMN].quantile([0.005, 0.995])
        df = df[df[TARGET_COLUMN].between(low, high)]
    return df

## 3. Feature Engineering

In [20]:
def feature_engineering(df):
    df = df.copy()
    engine = df["engine"].astype(str)
    df["age"] = 2024 - df["model_year"]
    df["log_milage"] = np.log1p(df["milage"])
    df["hp"] = engine.str.extract(r"(\d+\.?\d*)HP", expand=False).astype(float)
    df["liters"] = engine.str.extract(r"(\d+\.?\d*)\s*L(?:iter)?\b", expand=False).astype(float)
    df["cylinders"] = engine.str.extract(r"(\d+) Cylinder|\b[VIHW](\d{1,2})\b").astype(float).bfill(axis=1).iloc[:, 0]
    df["automatic"] = df["transmission"].astype(str).str.contains(r"A/T|Auto|CVT", case=False).astype(int)
    df["accident_flag"] = df["accident"].astype(str).str.contains("At least").astype(int)
    df["clean_title_flag"] = df["clean_title"].eq("Yes").astype(int)
    return df

## 4. Transform, Normalize, and Scale Features

In [21]:
NUM_COLS = ["age", "log_milage", "hp", "liters", "cylinders", "automatic", "accident_flag", "clean_title_flag"]
CAT_COLS = ["brand", "model", "fuel_type", "ext_col", "int_col"]
preprocessor = None


def scale_features(df):
    global preprocessor
    X = df[NUM_COLS + CAT_COLS]
    # Fitted on the first call (training data), reused afterwards
    if preprocessor is None:
        preprocessor = ColumnTransformer([
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM_COLS),
            ("cat", Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5)),
            ]), CAT_COLS),
        ]).fit(X)
    return preprocessor.transform(X)

In [22]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)
    return df

## 5. Train Test split

In [23]:
def split_data(df):
    train_split, temp = train_test_split(df, test_size=0.3, random_state=42)
    validation_split, test_split = train_test_split(temp, test_size=0.5, random_state=42)
    return train_split, validation_split, test_split


df_clean = data_cleaning(df)
train_df, val_df, test_df = split_data(df_clean)
print(len(train_df), len(val_df), len(test_df))

2221 476 476


## 6. Model Training and Hyperparameter Tuning


In [ ]:
def evaluate_predictions(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

In [25]:
preprocessor = None
X_train = preprocess_data(train_df.drop(columns=TARGET_COLUMN))
X_val = preprocess_data(val_df.drop(columns=TARGET_COLUMN))
y_train, y_val = train_df[TARGET_COLUMN], val_df[TARGET_COLUMN]


def build_model(alpha):
    return TransformedTargetRegressor(regressor=Ridge(alpha=alpha), func=np.log1p, inverse_func=np.expm1)


results = {}
for alpha in [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]:
    m = build_model(alpha).fit(X_train, y_train)
    results[alpha] = np.sqrt(mean_squared_error(y_val, m.predict(X_val)))
print(pd.Series(results, name="val_RMSE"))

best_alpha = min(results, key=results.get)
model = build_model(best_alpha).fit(X_train, y_train)
print("Best alpha:", best_alpha)

print("\nTrain")
evaluate_predictions(y_train, model.predict(X_train))
print("\nValidation")
evaluate_predictions(y_val, model.predict(X_val));

0.003     17695.895619
0.010     17697.092498
0.030     17700.761638
0.100     17707.908025
0.300     17740.403798
1.000     17935.585330
3.000     18538.517459
10.000    19965.689970
Name: val_RMSE, dtype: float64
Best alpha: 0.003

Train
--- Metrics ---
      RMSE: 19343.1536
       MAE: 9728.9361
        R2: 0.7682

Validation
--- Metrics ---
      RMSE: 17695.8956
       MAE: 9417.5203
        R2: 0.7162


In [26]:
def predict_price(df):
    processed = preprocess_data(df)
    predictions = model.predict(processed)
    return predictions

## 7. Evaluate on the Held-Out Test Set

In [27]:
evaluate_predictions(test_df[TARGET_COLUMN], predict_price(test_df.drop(columns=TARGET_COLUMN)));

--- Metrics ---
      RMSE: 24734.9548
       MAE: 11696.9292
        R2: 0.6679
